# 03 — Stage 2 Sketch

**AlphaMotion as the DINO of motion** — Stage 2 design document.

Stage 1 (NB02) validated the trunk architecture mechanically: pair-first +
OPM-as-gate + FM head + 4-component loss + per-clip variance probe.
Stage 2 asks the foundational question:

> Does the trunk, **scaled up**, learn motion representations that
> downstream tasks (editor, shadow imitation, fighting reaction, …) can
> use without retraining the trunk?

This is the same question DINO asked for vision: do unsupervised features
generalize to depth estimation, segmentation, video diffusion?

| | |
|---|---|
| Status   | design + pseudocode + protocol; no runs yet |
| Trains   | ❌ — Stage 1 6-variant full epoch must finish first |
| Linked   | `note/motionformer-research-vision.md` §10+ (to be written) |
| Resource | `notebook/03_resources/` |

## Outline
- §0  Bootstrap + load NB1 config
- §1  What AlphaMotion is — DINO of motion (middle-stream)
- §2  Why we explicitly **reject** caption / language / contrastive supervision
- §3  Motor-similarity probe — the core eval protocol (no labels required)
- §4  Scaling experiment design (params × data × steps sweep)
- §5  Gradient guidance — turning trunk into editor + controller
- §6  Stage 2 milestones (revised)
- §7  Out-of-scope and explicit non-goals


## §0 Bootstrap


In [ ]:
# §0.1 sys.path + load shared NB1 config (single source of truth)
import sys, json
from pathlib import Path

REPO_ROOT = Path.cwd().parent
RESOURCES = Path("03_resources")
NB1_CFG   = Path("01_resources/configs")

sys.path.insert(0, str(Path("01_resources/scripts")))   # model + data
sys.path.insert(0, str(Path("02_resources/scripts")))   # eval helpers (FM regime)
sys.path.insert(0, str(RESOURCES / "scripts"))          # Stage-2-specific scripts (TBD)

ENV      = json.loads((NB1_CFG / "env.json").read_text())
DATA     = json.loads((NB1_CFG / "data.json").read_text())
VARIANTS = json.loads((NB1_CFG / "variants.json").read_text())
print(f"loaded NB1 config — variants: {list(VARIANTS.keys())}")


## §1 What AlphaMotion is — DINO of motion

| | DINO (vision)                       | AlphaMotion (motion)                       |
|---|---|---|
| Input            | RGB image                           | SOMA-77 motion sequence                    |
| Trunk objective  | self-supervised representation      | self-supervised representation             |
| Trunk output     | per-patch CLS-style features        | per-(t, j) MSA hidden + pair tensor [J,J]  |
| **Downstream**   | depth / Gaussian splatting / video diffusion / classification — **task head trained on top of frozen trunk** | shadow imitation / fighting reaction / motion editor / control — **task head + cost function on frozen trunk** |
| Stage 1 thesis   | features beat supervised pretraining | pair-first features beat token-stream (Kimodo / HY / SONIC) |
| Stage 2 thesis   | scale unlocks emergent capabilities | **scale unlocks motion-style emergence** (manipulation vs locomotion separable in latent) |

**Positioning**:
- NOT upstream: AlphaMotion does not reason ("brain") — VL / world-model does
- NOT downstream: AlphaMotion does not directly produce motor commands ("muscle")
- **Middle-stream / cerebellum**: AlphaMotion encodes motion structure that
  upstream reasoning can constrain (via gradient guidance) and downstream
  tracking (PPO in Isaac, Stage 3) can execute

The key claim that distinguishes us from all motion generators in the
literature: **the trunk is the product**, not the head, not the loss, not
the application.


## §2 Why we explicitly reject caption / language / contrastive supervision

A common temptation in motion modeling is to add a contrastive loss with
caption embeddings (CLIP-text, T5, etc.). **For AlphaMotion this is wrong**.

### Motion vs language information density

Language is **lossy** about motion. Concrete pairs of motions that share
identical captions but differ markedly:

| Caption | Motion variant 1 | Motion variant 2 |
|---|---|---|
| "walk forward" | toes-out (外八字), narrow stance | toes-in (内八字), wide stance |
| "fold arms"    | calm cross at chest             | tense, pulled-up shoulders (defiance) |
| "raise hand"   | slow controlled lift            | sudden high reflex |
| "kick"         | front-thrust kick               | round-house kick |

If we train trunk with `contrastive(motion_emb, caption_emb)`, all these
pairs get the **same target embedding** → trunk is **forced to forget**
the motor distinctions that matter for downstream tasks.

### What we use instead
- Self-supervised **structure** loss (FM velocity matching across t)
- **Reconstruction** loss (FK consistency, smoothness, joint limits)
- **Multi-view temporal augmentation** (different temporal crops of the
  same clip should be embedding-close — NOT caption-supervised, just
  structure-of-the-motion supervised)

The trunk learns motion-internal structure. Upstream VL/WM gives semantic
context **at inference time** via gradient-guidance cost (see §5), not
during training.


## §3 Motor-similarity probe — core eval protocol

This is the Stage 2 main metric. Adapted from `experiment/stage1/pair_distance_probe.py`
(your earlier work, before deterministic-regression bug). Reframed for FM regime.

### Why motor-pair distance, not silhouette / ARI / NMI

- Silhouette / ARI / NMI all need **action labels**, which we don't trust
  (12-family regex, 58% AMASS = "other", semantic-loaded)
- Motor-pair distance needs only **hand-curated pairs** that you (the
  researcher) judge to be motorically similar / different
- The pair list is small (~50 pairs) and inspectable
- Tests the **directional hypothesis** explicitly:
  `dist(same_clip)  <  dist(similar_motor)  <  dist(different_motor)`

If this inequality holds at scale, the trunk has learned motion structure
that aligns with motor reality — without ever seeing a label.


In [ ]:
# §3.1 Motor-similarity pair list (HELPER — to be expanded)
#
# Each tuple = (prompt_keyword_a, prompt_keyword_b, category)
# 'same'      — same Kimodo prompt, different sample (different generation noise)
# 'similar'   — different prompts, motor-similar (locomotion variants, reach variants)
# 'different' — different prompts, motor-distinct (locomotion vs manipulation)
#
# Source: experiment/stage1/pair_distance_probe.py SIMILAR_MOTOR_PAIRS

SIMILAR_MOTOR_PAIRS = [
    # locomotion family
    ("walks forward",          "walks backward"),
    ("walks forward",          "jogs slowly"),
    ("runs in a straight line", "sprints forward"),
    # right-arm reach / grasp family
    ("picks up a cup with the right hand",  "reaches up to a high shelf"),
    ("picks up a cup with the right hand",  "drinks from a cup"),
    ("opens a door with the right hand",    "picks up a cup with the right hand"),
    # TODO: extend — aim ~30-50 pairs across locomotion / manipulation / stance / dance / martial
]

DIFFERENT_MOTOR_PAIRS = [
    # locomotion vs manipulation (very different motor strategy)
    ("walks forward",          "picks up a cup with the right hand"),
    ("runs in a straight line", "opens a door with the right hand"),
    # locomotion vs stance
    ("walks forward",          "sits down on a chair"),
    # TODO: extend — aim ~30-50 pairs
]


In [ ]:
# §3.2 Embedding extractor (FM regime — adapt from eval_style_emerg.extract_pair_embeddings)
#
# Use the SAME methodology as NB02 §3 (pair_norm hook on last block, t=1
# data state). Returns one embedding per clip.
#
# IMPLEMENTATION (to live in 03_resources/scripts/motor_pair_probe.py):
#
#   def extract_motion_embeddings(
#       ckpt_path: Path,
#       subset,                          # Subset of MixedMotionDataset
#       *, embedding: str = 'pair_norm', # 'pair_norm' | 'msa_pool'
#       device: str = 'cuda',
#   ) -> tuple[np.ndarray, list[str]]:
#       # Returns (X[N, D], prompts[N])
#       # Reuses _extract_pair / _extract_msa_pool from eval_style_emerg.py
#       ...


In [ ]:
# §3.3 Distance-ratio metric — the Stage 2 main number
#
# IMPLEMENTATION:
#
#   def motor_pair_distance_test(X, prompts, similar_pairs, different_pairs):
#       # For each pair (p_a, p_b):
#       #   collect embeddings whose prompts match keyword p_a or p_b
#       #   compute mean pairwise cosine distance across the two groups
#       # Returns dict:
#       #   {
#       #     'mean_same':       float,  # same prompt, different noise
#       #     'mean_similar':    float,  # similar motor strategy
#       #     'mean_different':  float,  # different motor strategy
#       #     'ratio_diff/sim':  float,  # >1 = motor structure learned
#       #     'ratio_sim/same':  float,  # >1 = within-motor sample variation captured
#       #     'p_value_diff_vs_sim': float,  # paired t-test
#       #   }
#
# Headline number for Stage 2 paper:
#   ratio = mean_different / mean_similar
#   1.0  = no motor structure learned (pure noise)
#   >1.5 = clear motor structure
#   >2.5 = strong, paper-worthy emergence


### §3.4 Interpreting results

What we want to see as scale increases:
- `mean_same  < mean_similar  < mean_different`  (strict ordering)
- `ratio_diff/sim` grows monotonically with (params × data × steps)
- Effect appears in `full` and `opm_only` (input-conditioned variants)
- Effect absent in `triangle_only` and `pair_static` (no input conditioning,
  per Stage 1 PCA collapse evidence)

What disqualifies the thesis:
- `ratio_diff/sim ≈ 1` even at largest scale → pair tensor doesn't capture
  motor structure → AlphaMotion is **not** the DINO of motion → architecture
  needs rethinking
- Effect appears equally in `pair_static` → the pair tensor isn't doing the
  work; positional encodings + transformer alone suffice


## §4 Scaling experiment design

The empirical question driving Stage 2:

> At what scale does motion-style emergence (§3 ratio) become visible?

Following DINO's scaling-law style figure: vary one axis at a time, plot
the §3 main metric.


In [ ]:
# §4.1 4-point sweep config
#
# Compute budget assumption: ~3 weeks of GPU time (1 RTX PRO 6000 96GB).
# Each cell of the matrix is one full training run.
#
# Axis 1: model size (params)
# Axis 2: data volume (clips)
# Axis 3: training steps
#
# To keep the experiment tractable we do NOT do a full 3D sweep — instead:
#   (a) anchor at small (smoke-current), grow each axis separately
#   (b) jump to satellite "max budget" point
#
# This is enough for a scaling-law plot.

SCALE_GRID = [
    # name              params   clips    steps   wall_time_estimate
    ("smoke",          "2.4M",  "5k",    "1k",   "10 min"),    # current baseline
    ("data_axis",      "2.4M",  "100k",  "30k",  "~6 h"),      # bigger data, same model
    ("model_axis",     "30M",   "5k",    "30k",  "~12 h"),     # bigger model, same data
    ("max_budget",     "30M",   "100k",  "30k",  "~24-36 h"),  # both axes together
]

# Each row of SCALE_GRID is one training config + one §3 motor-pair metric.
# Plot: x = log(params * clips * steps),  y = ratio_diff/sim → scaling law

# IMPLEMENTATION (to live in 03_resources/scripts/scale_sweep.py):
#
#   def run_scale_grid(grid: list[tuple], variants_subset: list[str], out_root: Path):
#       # For each (name, params, clips, steps):
#       #   generate a config + launch training (reuses train_runner.py)
#       #   wait for completion
#       #   evaluate §3 motor-pair distance
#       #   record into out_root / 'scale_results.json'
#       ...


### §4.2 Data integration — captions kept as analysis reference, NOT training signal

We need ~100k motion clips (DINO-comparable scale). Sources, all SMPL/SOMA-77 compatible:

| dataset           | clips    | captions in trunk loss? | captions kept on disk? | role |
|---|---|---|---|---|
| AMASS canonical   | 7.3k     | NO                      | (no captions)           | already integrated; diverse mocap |
| HumanML3D         | ~14k     | **NO**                  | **YES — for analysis**  | motion volume + caption sanity column |
| Motion-X          | ~80k     | **NO**                  | **YES — for analysis**  | bulk volume + caption sanity column |

**Strict separation between training signal and analysis reference**:

- **Trunk training**: pure motion-only self-supervision (FM + 4-loss + Tier 2
  kinematic priors). Caption fields are NOT in the loss path. See §2 for the
  information-theoretic argument.

- **Caption usage (analysis only)**:
  * Sanity-check the §3 motor-pair list — do clips that share the same
    caption tend to land in the same motor cluster? If they don't, our
    motor pair list may need revision.
  * Color overlays on motor-similarity scatter plots — useful for paper
    figures even if captions weren't training input.
  * Diagnostic: spot clips whose embedding is far from caption-cluster
    centroid → potential annotation errors / OOD examples.
  * Cross-validation against language-based prior work (compare our motor
    cluster purity against caption cluster purity for the same clips).

Captions are **inputs to the eval / diagnostic pipeline, not the model**.
The model sees only motion. The researcher (and reader) sees captions to
interpret what the model learned.

Integration work:
1. Schema unification — all sources to SOMA-77 skeleton + 30 fps + clip_len 90
2. Manifest: `data/mixed_extended.npz` with motion fields + `caption` column
   stored separately (e.g. `captions.json` keyed by clip id)
3. New `splits.json` with stratified subject-wise heldout
4. Re-run Stage 1 6-variant smoke on extended data to verify pipeline

This is **Stage 2 prep** — not part of the main scaling experiment.


### §4.3 Model scaling — keep architecture, scale dimensions

All variants stay the same (full / opm_only / triangle_only / pair_static /
axial_only / baseline). We scale per-cell along three model knobs:

| name      | depth | hidden | pair_hidden | heads | param count (full) |
|---|---|---|---|---|---|
| smoke     | 6     | 128    | 32          | 4     | ~2.4M              |
| medium    | 8     | 256    | 64          | 8     | ~10M               |
| large     | 12    | 384    | 96          | 12    | ~30M               |
| xlarge    | 16    | 512    | 128         | 16    | ~80M               |

Implementation: just edit `configs/variants.json` (NB1 §0.5 dump) per scale point.
Trunk code does not change.


## §5 Gradient guidance — turning trunk into editor + controller

The trunk + FM head together define a learned vector field `v_θ(x_t, t)`.
At inference, classifier guidance lets us **steer** sampling toward
arbitrary differentiable cost functions, **without retraining**.

This is the same mechanism BeyondMimic Stage 2 uses. It's what makes the
trunk genuinely **task-agnostic** — one trunk, many cost functions.


In [ ]:
# §5.1 FM classifier guidance — pseudo-code
#
# Standard FM Euler integration:
#     for k in range(N): x_{t+dt} = x_t + dt * v_θ(x_t, t)
#
# Guided FM Euler integration:
#     for k in range(N):
#         v   = v_θ(x_t, t)
#         g   = ∇_x  cost(x_t, t)              # task-specific cost gradient
#         x_{t+dt} = x_t + dt * (v - λ(t) * g)  # subtract gradient direction
#
# λ(t) schedule: typically grows with t (more guidance near data manifold)
#
# IMPLEMENTATION (to live in 03_resources/scripts/gradient_guidance.py):
#
#   def fm_sample_with_guidance(
#       model,
#       cost_fn: Callable[[torch.Tensor, torch.Tensor, dict], torch.Tensor],
#       cost_kwargs: dict,
#       *, n_steps: int = 16, lam_schedule: Callable[[float], float] = lambda t: t,
#   ) -> tuple[torch.Tensor, ...]:
#       # Standard noise → integrate v_θ from t=0 to t=1, applying gradient
#       # of cost_fn at each step. Returns (x_pos_1, x_rot_1, x_root_1).
#       ...


### §5.2 Cost function library

The trunk + guidance form a generic motion editor. Each application is one
`cost_fn`:

| Application       | cost_fn(x_t, t) signature                                      |
|---|---|
| Keyframe constraint | `‖FK(x̂_endpoint)[t_key] − target_pose‖²`                    |
| End-effector goal   | `‖FK(x̂).right_hand − target_xyz‖²`                          |
| Joystick velocity   | `‖root_vel(x̂) − command_vel‖²`                              |
| Obstacle avoidance  | `max(0, buffer_radius − dist_to_obstacle(x̂))²`              |
| Style transfer      | `‖MSA_pool(x̂_endpoint) − reference_style_embedding‖²`       |
| **Shadow imitation** | `‖x̂_endpoint − live_human_motion_input(t)‖²`              |
| **Counter-strategy** | `−adversarial_advantage(x̂, opponent_motion)`              |
| VL / WM cue          | `cosine_dist(MSA_pool(x̂), VL_reasoning_embedding)`         |

The whole trunk + head is **frozen**. Each application is one ~50-line
`cost_fn` definition + a tuned `λ(t)` schedule.


In [ ]:
# §5.3 Shadow imitation demo — pseudo-code (Stage 2 transfer demo target)
#
# Setup:
#   - Robot (Unitree G1 / similar) is observed via VR / video / mocap
#   - Live human motion arrives as SMPL pose stream (frame-by-frame)
#   - Goal: robot mirrors human motion in real time
#
# Naive approach: direct retargeting (resmimic style) — pose-to-pose copy
# Our approach: AlphaMotion identifies which joints to activate,
#               cost = distance to live human input,
#               trunk autonomously fills in coordination + smoothness
#
# IMPLEMENTATION sketch:
#
#   def shadow_imitation_step(
#       trunk_model,
#       human_motion_window,    # last K frames of human SMPL pose
#   ) -> robot_target_pose:
#       def cost_fn(x_t, t, kwargs):
#           # encode human window, compare to model's predicted endpoint
#           pred_endpoint = endpoint_extrapolate(x_t, t, model)
#           human_target  = retarget_human_to_robot(human_motion_window)
#           return torch.mean((pred_endpoint - human_target) ** 2)
#
#       x_pos, x_rot, x_root = fm_sample_with_guidance(
#           trunk_model, cost_fn, cost_kwargs={...},
#           n_steps=8,                # cheap for real-time
#       )
#       return x_pos, x_rot, x_root   # send to PD controller
#
# Real-time requirement: 30 Hz robot control → each guided sample budget
# ~33 ms. With 8 FM steps × ~3 ms forward each = ~25 ms. Feasible.


### §5.4 Stage 3 preview — production-scale deployment (the VC-pitch level)

Stage 3 is **production-grade**: full data + full model + PPO finetune + real
robot deployment. The compute budget is **128× H100 cluster** (vs Stage 2's
single RTX PRO 6000). This is what the funding pitch is built around.

**Stage 3 components** (each is a multi-week effort at production scale):

| Component | What it adds | Compute scale |
|---|---|---|
| **Massive trunk scaling** | Stage-2 architecture at 1B+ params, 1M+ motion clips, 100k+ steps | 128 H100 × weeks |
| **Action head + PPO finetune** | Add PD-setpoint head; PPO in Isaac Lab to learn dynamics-stable execution | sim2real, parallel envs |
| **State-action co-diffusion** (BeyondMimic-style) | Frozen trunk + jointly-modeled (state, action) latent diffusion → guidance steers both | 1 H100 cluster sub-job |
| **Multi-modality input adapters** | VL / video / VR / audio cost functions for §5 guidance, plugged into universal token space (SONIC-style) | depends on modality |
| **Real-robot deployment loop** | Unitree G1 / similar; closed-loop control + safety wrappers; iterative sim2real refinement | hardware-bound |
| **Generation pipeline tooling** | Inference server + caching + low-latency sampling for productized motion editor | engineering scale |

**Stage 2 → Stage 3 architecture continuity**:

```
trunk MSA → ┬→ velocity head      (Stage 1-2: kinematic motion)
            ├→ contact head       (Stage 2 optional: contact prediction)
            ├→ COM head           (Stage 2 optional: balance signal)
            └→ ACTION head        ← Stage 3: PD setpoints
                                    PPO-trained on top of frozen trunk
```

**Stage 2 design decision that enables this**: the trunk's MSA hidden state
must be **rich enough** for an action head to read off PD setpoints. The §3
motor-pair probe is the scaled-emergence signal that says "yes, this trunk
will support a 1B-param Stage-3 trunk being built on the same architecture".

**Why this is the VC story**: production-grade humanoid foundation model
trained at GPT-3 scale, end-to-end from raw motion data → real robot motion,
with the same trunk reused across diverse downstream applications via
gradient guidance. Stage 1 + Stage 2 together prove the architecture
scales; Stage 3 is the bet on building the actual product.


## §6 Stage 2 milestones (revised)

| ID  | Milestone | Definition of done |
|---|---|---|
| **M1** | Stage 1 6-variant full epoch | All variants converged at 30k+ steps; smoke replaced with real numbers in NB02 headline |
| **M2** | Motor-pair probe pipeline | `motor_pair_probe.py` implemented; baseline numbers on Stage 1 ckpts logged |
| **M3** | Extended dataset | `data/mixed_extended.npz` with AMASS + HumanML3D + Motion-X (no captions); ~100k clips total |
| **M4** | Scaling experiment | 4-point grid trained; ratio_diff/sim plotted vs compute |
| **M5** | Gradient guidance v1 | `fm_sample_with_guidance` implemented; keyframe demo works on Stage 1 ckpt |
| **M6** | Transfer demo: shadow imitation | One downstream task end-to-end with frozen trunk + small head + cost function |
| **M7** | Stage 2 paper-ready figure | scaling law (M4) + transfer demo (M6) packaged as headline |

M1 must finish before everything else (smoke ckpts insufficient for any
emergence claim). M2 depends on M1. M3 + M4 are the heavy compute. M5 + M6
can start in parallel with M3 since they only need a frozen Stage-1 ckpt.


## §7 Out-of-scope and explicit non-goals

What Stage 2 is **NOT** doing:

1. **No caption / language / contrastive supervision** — see §2 for the
   information-theoretic argument. Motion ≠ language.
2. **No PPO / Isaac Lab integration** — that's Stage 3 (production-scale,
   128× H100 budget). Stage 2 produces kinematic motion only; physical
   stability + real robot deployment lives in Stage 3.
3. **No π0 / Pi-Zero adoption** — we target *beating* Pi, not adopting it.
4. **No two-stage (root + body) decoder** like Kimodo — our pair-first
   trunk handles joint coupling architecturally; no need for Kimodo-style
   structural decomposition in the head.
5. **No MMDiT** like HY-Motion — we don't have a separate text stream;
   text/VL only enters at inference via gradient guidance, not as a
   parallel attention stream.
6. **No multi-modal contrastive (motion ↔ image / video)** — same reason as
   §2: imposes external semantic structure on the trunk's representation.

Open questions for **future** stages (post-Stage-2):
- Does freezing the trunk + adding an action head + PPO finetune give
  better tracking than SONIC's universal policy?
- Does scaling beyond 30M params show emergent capabilities Pi-Zero
  doesn't have?
- Can we replace the MoCap data dependency with cross-embodiment learning
  (human video → motion via vision foundation model)?
